In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("PortfolioValues").getOrCreate()

portfolio = spark.createDataFrame(
    [
        ("Alpha", "A", 1000),
        ("Alpha", "B", 2000),
        ("Beta", "A", 1500),
        ("Beta", "C", 2500),
        ("Gamma", "B", 1200),
        ("Gamma", "C", 1300),
    ],
    ["PE_firm", "company", "shares"],
)

prices = spark.createDataFrame(
    [
        ("2023-01-01", "A", 50.0),
        ("2023-01-01", "B", 20.0),
        ("2023-01-01", "C", 30.0),
        ("2023-01-02", "A", 52.0),
        ("2023-01-02", "B", 21.0),
        ("2023-01-02", "C", 31.0),
    ],
    ["date", "company", "closing_price"],
)

portfolio.show()
prices.show()

+-------+-------+------+
|PE_firm|company|shares|
+-------+-------+------+
|  Alpha|      A|  1000|
|  Alpha|      B|  2000|
|   Beta|      A|  1500|
|   Beta|      C|  2500|
|  Gamma|      B|  1200|
|  Gamma|      C|  1300|
+-------+-------+------+

+----------+-------+-------------+
|      date|company|closing_price|
+----------+-------+-------------+
|2023-01-01|      A|         50.0|
|2023-01-01|      B|         20.0|
|2023-01-01|      C|         30.0|
|2023-01-02|      A|         52.0|
|2023-01-02|      B|         21.0|
|2023-01-02|      C|         31.0|
+----------+-------+-------------+



In [2]:
merged_df = portfolio.join(prices, on="company", how="inner")

# Compute portfolio value for each PE_firm and date
portfolio_value = merged_df.withColumn(
    "value",
    col("shares") * col("closing_price"),
)
portfolio_value.show()

+-------+-------+------+----------+-------------+-------+
|company|PE_firm|shares|      date|closing_price|  value|
+-------+-------+------+----------+-------------+-------+
|      A|  Alpha|  1000|2023-01-01|         50.0|50000.0|
|      A|  Alpha|  1000|2023-01-02|         52.0|52000.0|
|      A|   Beta|  1500|2023-01-01|         50.0|75000.0|
|      A|   Beta|  1500|2023-01-02|         52.0|78000.0|
|      B|  Alpha|  2000|2023-01-01|         20.0|40000.0|
|      B|  Alpha|  2000|2023-01-02|         21.0|42000.0|
|      B|  Gamma|  1200|2023-01-01|         20.0|24000.0|
|      B|  Gamma|  1200|2023-01-02|         21.0|25200.0|
|      C|   Beta|  2500|2023-01-01|         30.0|75000.0|
|      C|   Beta|  2500|2023-01-02|         31.0|77500.0|
|      C|  Gamma|  1300|2023-01-01|         30.0|39000.0|
|      C|  Gamma|  1300|2023-01-02|         31.0|40300.0|
+-------+-------+------+----------+-------------+-------+



In [3]:
# Group by PE_firm and date and sum up the value
portfolio_value = portfolio_value.groupby(["PE_firm", "date"]).agg(
    sum("value").alias("portfolio_value")
)


portfolio_value.show()

+-------+----------+---------------+
|PE_firm|      date|portfolio_value|
+-------+----------+---------------+
|  Gamma|2023-01-02|        65500.0|
|  Alpha|2023-01-02|        94000.0|
|  Alpha|2023-01-01|        90000.0|
|   Beta|2023-01-02|       155500.0|
|   Beta|2023-01-01|       150000.0|
|  Gamma|2023-01-01|        63000.0|
+-------+----------+---------------+

